# Adaptive Edge V2.1 — low learning rate fine-tuning

Fine-tune the **original gated V2 best checkpoint** in two independent runs. The original 100 LOL-v2 Real Test pairs are reused for validation and checkpoint selection; results are **not held-out test estimates**. Full training cells are opt-in and can take many GPU hours. Do not run the original full-training notebook for this experiment.

## Environment setup

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()), flush=True)

In [ ]:
%cd /content
!if [ ! -d /content/myRetinexformer/.git ]; then git clone https://github.com/Silly-Gooses/myRetinexformer.git /content/myRetinexformer; fi
import subprocess
subprocess.run(["git", "fetch", "origin", "v2.1"], cwd="/content/myRetinexformer", check=True)
subprocess.run(["git", "switch", "v2.1"], cwd="/content/myRetinexformer", check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "v2.1"], cwd="/content/myRetinexformer", check=True)
%cd /content/myRetinexformer
from pathlib import Path
assert Path("finetune_v2_1.py").is_file(), "Push the v2.1 branch before running Colab"

In [ ]:
!pip install -q einops opencv-python pillow matplotlib

## Google Drive and experiment configuration

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess, sys
from finetune_v2_1 import source_info, settings_for

DATASET_ROOT = Path("/content/drive/MyDrive/datasets/LOLv2/Real_captured")
OUTPUT_ROOT = Path("/content/drive/MyDrive/RetinexFormer_full_training")
ORIGINAL_ROOT = OUTPUT_ROOT / "Proposed_RetinexFormer_V2_E3_Gated"
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
MAX_ADDITIONAL_EPOCHS = 30
VALIDATION_INTERVAL_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 3
MIN_PSNR_IMPROVEMENT = 0.01

# For each experiment, set its own resume flag after a Colab disconnect.
RESUME_LR5 = False
RESUME_LR10 = False

assert ORIGINAL_ROOT != OUTPUT_ROOT / "v2.1"
print("Original:", ORIGINAL_ROOT)
print("Fine-tuning outputs:", OUTPUT_ROOT / "v2.1")

## Original best checkpoint and Drive preflight

In [ ]:
from finetune_v2_1 import source_info, settings_for
from official_workflow import storage_probe
import json
SOURCE_INFO, SOURCE_CONFIG, SOURCE_MANIFEST, _ = source_info(ORIGINAL_ROOT, "cpu")
assert Path(SOURCE_CONFIG["dataset_root"]) == DATASET_ROOT, "Dataset path differs from checkpoint"
print(json.dumps(SOURCE_INFO, indent=2), flush=True)
print("LR / 5:", settings_for(SOURCE_CONFIG, "finetune_lr5"), flush=True)
print("LR / 10:", settings_for(SOURCE_CONFIG, "finetune_lr10"), flush=True)
print("Drive storage probe:", storage_probe(OUTPUT_ROOT), flush=True)

## Isolated two-update smoke test

Run this before either full experiment. It loads only 16 training pairs and 2 small validation images from the original manifest. Its scores are diagnostic and cannot be compared with full-set validation. It creates a separate timestamped directory on Drive and never writes to the original checkpoint or production fine-tuning folders.

In [ ]:
RUN_SMOKE = False  # @param {type:"boolean"}
if RUN_SMOKE:
    import os
    import torch
    from datetime import datetime, timezone
    if not torch.cuda.is_available():
        raise RuntimeError("Smoke test requires a CUDA GPU in Colab; select an A100 runtime")
    print("Smoke device:", torch.cuda.get_device_name(torch.cuda.current_device()), flush=True)
    smoke_parent = OUTPUT_ROOT / ("V2_1_Smoke_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
    command = [sys.executable, "-u", "finetune_v2_1.py", "--original-root", str(ORIGINAL_ROOT),
               "--output-root", str(smoke_parent), "--device", "cuda",
               "--experiment", "finetune_lr5", "--smoke"]
    print("Launching smoke with direct, unbuffered cell output:", command, flush=True)
    subprocess.run(command, check=True, stdout=None, stderr=None,
                   env=dict(os.environ, PYTHONUNBUFFERED="1"))
    print("Smoke output:", smoke_parent / "v2.1" / "smoke" / "finetune_lr5", flush=True)

## Fine-tuning LR / 5

Starts from the original best model weights. `--resume` restores this run’s own latest optimizer, scaler, RNG, and progress after a disconnect.

In [ ]:
RUN_LR5 = False  # @param {type:"boolean"}
if RUN_LR5:
    cmd = [sys.executable, "-u", "finetune_v2_1.py", "--original-root", str(ORIGINAL_ROOT),
           "--output-root", str(OUTPUT_ROOT), "--device", DEVICE, "--experiment", "finetune_lr5",
           "--max-additional-epochs", str(MAX_ADDITIONAL_EPOCHS),
           "--validation-interval-epochs", str(VALIDATION_INTERVAL_EPOCHS),
           "--patience", str(EARLY_STOPPING_PATIENCE),
           "--min-improvement", str(MIN_PSNR_IMPROVEMENT)]
    if RESUME_LR5: cmd.append("--resume")
    subprocess.run(cmd, check=True)

## Fine-tuning LR / 10

This independently loads the **same original best snapshot**. It never initializes from LR / 5.

In [ ]:
RUN_LR10 = False  # @param {type:"boolean"}
if RUN_LR10:
    cmd = [sys.executable, "-u", "finetune_v2_1.py", "--original-root", str(ORIGINAL_ROOT),
           "--output-root", str(OUTPUT_ROOT), "--device", DEVICE, "--experiment", "finetune_lr10",
           "--max-additional-epochs", str(MAX_ADDITIONAL_EPOCHS),
           "--validation-interval-epochs", str(VALIDATION_INTERVAL_EPOCHS),
           "--patience", str(EARLY_STOPPING_PATIENCE),
           "--min-improvement", str(MIN_PSNR_IMPROVEMENT)]
    if RESUME_LR10: cmd.append("--resume")
    subprocess.run(cmd, check=True)

## Comparison and plots

This cell only reads completed run metadata. Missing experiments remain missing; no score is fabricated. Original reference scores come from checkpoint metadata, and fine-tuning scores come from measured validation.

In [ ]:
RUN_COMPARISON = False  # @param {type:"boolean"}
if RUN_COMPARISON:
    subprocess.run([sys.executable, "finetune_v2_1.py", "--original-root", str(ORIGINAL_ROOT),
                    "--output-root", str(OUTPUT_ROOT), "--compare"], check=True)
    import csv
    import matplotlib.pyplot as plt
    from IPython.display import display
    base = OUTPUT_ROOT / "v2.1"
    with (base / "comparison/comparison.csv").open(newline="") as f:
        comparison = list(csv.DictReader(f))
    display(comparison)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for experiment in ("finetune_lr5", "finetune_lr10"):
        path = base / experiment / "logs/finetune.csv"
        if not path.is_file(): continue
        with path.open(newline="") as f: rows = list(csv.DictReader(f))
        for ax, field in zip(axes.flat, ("train_loss", "validation_psnr", "validation_ssim", "learning_rate")):
            points = [(int(r["additional_epoch"]), float(r[field])) for r in rows if r[field]]
            if points: ax.plot(*zip(*points), marker="o", label=experiment)
            ax.set(xlabel="Additional epoch", ylabel=field.replace("_", " ").title())
            ax.grid(alpha=.25)
    axes[0, 1].axhline(SOURCE_INFO["baseline_psnr"], color="black", linestyle="--", label="Original best")
    axes[1, 0].axhline(SOURCE_INFO["baseline_ssim"], color="black", linestyle="--", label="Original best")
    for ax in axes.flat: ax.legend()
    fig.tight_layout()
    fig.savefig(base / "comparison/curves.png", dpi=150)
    plt.show()

## Interpreting the result

Use `v2.1/comparison/comparison.csv` and each run’s `selection.json` for measured results. A selected `additional_epoch` of 0 means fine-tuning did not beat the original checkpoint. The historical Test set is also the checkpoint-selection validation set, so no independent held-out claim is possible from these scores. The original checkpoint is never modified.